In [2]:
import pandas as pd
import numpy as np
import torch
import os
import kagglehub
import re
from collections import defaultdict

In [3]:
path = kagglehub.dataset_download("mohamedbakhet/amazon-books-reviews")

# Загрузка Books_rating
df = pd.read_csv(os.path.join(path, 'Books_rating.csv'))

In [4]:
df = df.rename(columns={
    'User_id': 'user_id',
    'Id': 'book_id',
    'Title': 'title',
    'review/score': 'rating',
    'review/time': 'date_read'
})
df['date_read'] = pd.to_datetime(df['date_read'], unit='s', errors='coerce')

# Загрузка books_data
books = pd.read_csv(os.path.join(path, 'books_data.csv'))
books = books.rename(columns={
    'Title': 'title'
})

In [5]:
# Объединяем по названию книги
df = df.merge(books[['title', 'authors', 'categories']], on='title', how='left')
print(f"Загружено {len(df):,} взаимодействий")
print(f"Пользователей: {df['user_id'].nunique():,}")
print(f"Книг: {df['book_id'].nunique():,}")
print(f"Колонки: {df.columns.tolist()}")

Загружено 3,000,000 взаимодействий
Пользователей: 1,008,972
Книг: 221,998
Колонки: ['book_id', 'title', 'Price', 'user_id', 'profileName', 'review/helpfulness', 'rating', 'date_read', 'review/summary', 'review/text', 'authors', 'categories']


In [6]:
print(f"Уникальных авторов ДО базовой очистки: {df['authors'].nunique()}")
print(f"Уникальных категорий ДО базовой очистки: {df['categories'].nunique()}")

# убираем квадратные скобки и кавычки, точки и запятые и пробелы
df['authors'] = df['authors'].apply(
    lambda x: x.strip("[]'").replace("'", "") if isinstance(x, str) else 'unknown'
)
df['categories'] = df['categories'].apply(
    lambda x: x.strip("[]'").replace("'", "") if isinstance(x, str) else 'unknown'
)

df['authors'] = df['authors'].str.strip().str.lower().str.replace('.', '', regex=False)
df['authors'] = df['authors'].str.replace(r'\s+', ' ', regex=True)

df['categories'] = df['categories'].str.strip().str.lower().str.replace('.', '', regex=False)
df['categories'] = df['categories'].str.replace(r'\s+', ' ', regex=True)

print(f"Уникальных авторов после базовой очистки: {df['authors'].nunique()}")
print(f"Уникальных категорий после базовой очистки: {df['categories'].nunique()}")

Уникальных авторов ДО базовой очистки: 127278
Уникальных категорий ДО базовой очистки: 10883
Уникальных авторов после базовой очистки: 126788
Уникальных категорий после базовой очистки: 10605


In [7]:
# Поиск неявных дубликатов по фамилии и инициалам

author_counts = df['authors'].value_counts()
author_list = [a for a in author_counts.index if a != 'unknown']

# разбиваем на слова и классифицируем
full_names = []
initial_names = []

for author in author_list:
    words = author.split()
    if len(words) < 2:
        continue
    if all(len(w) == 1 for w in words[:-1]):
        initial_names.append((author, words))
    else:
        full_names.append((author, words))

# cравниваем только полные имена с инициалами
for full_author, full_words in full_names:
    full_lastname = full_words[-1]
    full_initials = ''.join(w[0] for w in full_words[:-1])

    for init_author, init_words in initial_names:
        init_lastname = init_words[-1]

        if full_lastname != init_lastname:
            continue

        init_initials = ''.join(init_words[:-1])

        if full_initials.startswith(init_initials):
            cnt1 = author_counts[full_author]
            cnt2 = author_counts[init_author]
            if cnt1 + cnt2 > 500:
                print(f"'{full_author}' ({cnt1:,}) ↔ '{init_author}' ({cnt2:,})")
                print()

'john ronald reuel tolkien' (12,906) ↔ 'j r r tolkien' (37,280)

'robert louis stevenson' (6,525) ↔ 'r l stevenson' (713)

'herman melville' (5,754) ↔ 'h melville' (1,738)

'adam smith' (3,431) ↔ 'a smith' (1)

'alexander mccall smith' (1,678) ↔ 'a smith' (1)

'virginia woolf' (1,595) ↔ 'v woolf' (1)

'diana gabaldon' (1,464) ↔ 'd gabaldon' (1)

'brian herbert, kevin j anderson' (1,163) ↔ 'b anderson' (3)

'clive staples lewis' (1,009) ↔ 'c s lewis' (11,824)

'mary higgins clark' (951) ↔ 'm clark' (1)

'mary higgins clark' (951) ↔ 'm h clark' (1)

'lucy maud montgomery' (931) ↔ 'l m montgomery' (745)

'david herbert lawrence' (863) ↔ 'd h lawrence' (284)

'jonathan swift' (754) ↔ 'j swift' (2)

'gilbert keith chesterton' (718) ↔ 'g k chesterton' (528)

'terence hanbury white' (655) ↔ 't h white' (63)

'james d watson' (598) ↔ 'j d watson' (8)

'cecil scott forester' (563) ↔ 'c s forester' (767)

'harriet jacobs' (555) ↔ 'h jacobs' (1)

'michael phillips' (543) ↔ 'm phillips' (1)

'alan

In [8]:
# исправляем найденные дубликаты
author_mapping = {
    'john ronald reuel tolkien': 'j r r tolkien',
    'john ronald reuel tolkien, christopher tolkien': 'j r r tolkien',
    'r l stevenson': 'robert louis stevenson',
    'robert louis stevenson, fanny van de grift stevenson': 'robert louis stevenson',
    'h melville': 'herman melville',
    'a smith': 'adam smith',
    'v woolf': 'virginia woolf',
    'd gabaldon': 'diana gabaldon',
    'b anderson': 'brian herbert, kevin j anderson',
    'clive staples lewis': 'c s lewis',
    'clive s lewis': 'c s lewis',
    'carolyn sherwin bailey, clara m lewis': 'c s lewis',
    'm clark': 'mary higgins clark',
    'm h clark': 'mary higgins clark',
    'l m montgomery': 'lucy maud montgomery',
    'd h lawrence': 'david herbert lawrence',
    'j swift': 'jonathan swift',
    'g k chesterton': 'gilbert keith chesterton',
    't h white': 'terence hanbury white',
    'e b white': 'elwyn brooks white',
    'elwyn brooks white, katharine sergeant angell white': 'elwyn brooks white',
    'j d watson': 'james d watson',
    'cecil scott forester': 'c s forester',
    'h jacobs': 'harriet jacobs',
    'm phillips': 'michael phillips',
    'a a milne': 'alan alexander milne',
    'p g wodehouse': 'pelham grenville wodehouse',
    'p wodehouse': 'pelham grenville wodehouse',
    'h a rey, margret rey': 'h a rey',
    'jerome david salinger': 'j d salinger',
    'e m forster': 'edward morgan forster',
    'edward m forster': 'edward morgan forster',
    'h g wells': 'herbert george wells',
    'herbert g wells': 'herbert george wells',
    'r l (written by kathryn lance) stine': 'r l stine',
    'a j frost, richard russell': 'a j russell',
    'r h dana': 'richard henry dana',
}

df['authors'] = df['authors'].replace(author_mapping)
print(f"Уникальных авторов после маппинга: {df['authors'].nunique()}")

# заполняем пропуски
df['authors'] = df['authors'].fillna('unknown')
df['categories'] = df['categories'].fillna('unknown')

Уникальных авторов после маппинга: 126751


In [9]:
# дубликаты книг
def normalize_title(t):
    t = str(t).strip().lower()
    t = t.replace('&', 'and')
    t = re.sub(r'[\(\[].*?[\)\]]', '', t)
    t = re.sub(r'[^\w\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['title'] = df['title'].apply(normalize_title)

# Оставляем один ISBN для каждого нормализованного названия
title_to_canonical_id = df.groupby('title')['book_id'].first().to_dict()
df['book_id'] = df['title'].map(title_to_canonical_id)

print(f"Уникальных книг после нормализации названий: {df['book_id'].nunique()}")

Уникальных книг после нормализации названий: 200458


In [11]:
df = df.sort_values('date_read')

In [12]:
df['date_read'].tail(30)

68148     2013-03-03
1441032   2013-03-03
1910779   2013-03-03
101821    2013-03-04
1888684   2013-03-04
1829528   2013-03-04
717118    2013-03-04
2806366   2013-03-04
960829    2013-03-04
2232589   2013-03-04
1531279   2013-03-04
2049176   2013-03-04
2818736   2013-03-04
2572566   2013-03-04
2572565   2013-03-04
1017673   2013-03-04
263375    2013-03-04
960831    2013-03-04
1830740   2013-03-04
960830    2013-03-04
1564116   2013-03-04
2476991   2013-03-04
720412    2013-03-04
2130022   2013-03-04
528155    2013-03-04
2846063   2013-03-04
310909    2013-03-04
203132    2013-03-04
2389710   2013-03-04
788484    2013-03-04
Name: date_read, dtype: datetime64[ns]

In [13]:
def preprocess_data(df, min_user_books=5, max_seq_len=30,
                    val_ratio=0.15, test_ratio=0.15):

    # фильтрация
    filtered_rows = []
    for _, row in df.iterrows():
        filtered_rows.append({
            'uid': row['user_id'],
            'item_ids': row['book_id'],
            'author_ids': row['authors'],
            'category_ids': row['categories'],
            'date_read': row['date_read']
        })
    df = pd.DataFrame(filtered_rows)

     # Группировка по пользователям
    user_data = df.groupby('uid').agg({
        'item_ids': list,
        'author_ids': list,
        'category_ids': list,
        'date_read': list
    }).reset_index()

    user_data = user_data[user_data['item_ids'].apply(len) >= min_user_books]
    print(f"\nПосле фильтрации (>= {min_user_books} книг): {len(user_data)} пользователей")

   # маппинг ID в индексы
    all_items = set()
    all_authors = set()
    all_categories = set()
    for _, row in user_data.iterrows():
        all_items.update(row['item_ids'])
        all_authors.update(row['author_ids'])
        all_categories.update(row['category_ids'])

    item_id_to_idx = {item: i+1 for i, item in enumerate(sorted(all_items))}
    author_to_idx = {a: i+1 for i, a in enumerate(sorted(all_authors))}
    category_to_idx = {c: i+1 for i, c in enumerate(sorted(all_categories))}
    print(f"Уникальных книг: {len(item_id_to_idx)}")
    print(f"Уникальных авторов: {len(author_to_idx)}")
    print(f"Уникальных категорий: {len(category_to_idx)}")

    n_users = len(user_data)
    n_test = int(n_users * test_ratio)
    n_val = int(n_users * val_ratio)

    df_sorted = user_data.sort_values('uid').reset_index(drop=True)

    test_df = df_sorted.iloc[-n_test:].copy()
    val_df = df_sorted.iloc[-n_test-n_val:-n_test].copy()
    train_df = df_sorted.iloc[:-n_test-n_val].copy()
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # обрезка последовательностей и маппинг
    def map_and_truncate(data):
        result = []
        for _, row in data.iterrows():
            items = row['item_ids'][-max_seq_len:]
            authors = row['author_ids'][-max_seq_len:]
            categories = row['category_ids'][-max_seq_len:]

            mapped_items = [item_id_to_idx.get(item, 0) for item in items]
            mapped_authors = [author_to_idx.get(a, 0) for a in authors]
            mapped_categories = [category_to_idx.get(c, 0) for c in categories]

            result.append({
                'uid': row['uid'],
                'item_ids': mapped_items,
                'author_ids': mapped_authors,
                'category_ids': mapped_categories
            })
        return pd.DataFrame(result)

    train_df = map_and_truncate(train_df)
    val_df = map_and_truncate(val_df)
    test_df = map_and_truncate(test_df)

    return train_df, val_df, test_df, item_id_to_idx, author_to_idx, category_to_idx

In [14]:
train_df, val_df, test_df, item_id_to_idx, author_to_idx, category_to_idx = preprocess_data(
    df, min_user_books=5, max_seq_len=30
)


После фильтрации (>= 5 книг): 82805 пользователей
Уникальных книг: 119132
Уникальных авторов: 75214
Уникальных категорий: 6629
Train: 57965, Val: 12420, Test: 12420


In [15]:
def prepare_tensors(data, max_seq_len=30):
    # тензоры из DataFrame
    inputs, targets = [], []
    author_inputs, category_inputs = [], []

    for _, row in data.iterrows():
        items = row['item_ids']
        authors = row['author_ids']
        categories = row['category_ids']

        for i in range(1, len(items)):
            history = items[max(0, i - max_seq_len):i]
            padded = [0] * (max_seq_len - len(history)) + history
            inputs.append(padded)
            targets.append(items[i])

            auth_hist = authors[max(0, i - max_seq_len):i]
            auth_padded = [0] * (max_seq_len - len(auth_hist)) + auth_hist
            author_inputs.append(auth_padded)

            cat_hist = categories[max(0, i - max_seq_len):i]
            cat_padded = [0] * (max_seq_len - len(cat_hist)) + cat_hist
            category_inputs.append(cat_padded)

    return (torch.tensor(inputs, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long),
            torch.tensor(author_inputs, dtype=torch.long),
            torch.tensor(category_inputs, dtype=torch.long))

In [16]:
cnt_item = len(item_id_to_idx)
cnt_author = len(author_to_idx)
cnt_category = len(category_to_idx)

train_inputs, train_targets, train_authors, train_categories = prepare_tensors(train_df)
val_inputs, val_targets, val_authors, val_categories = prepare_tensors(val_df)
test_inputs, test_targets, test_authors, test_categories = prepare_tensors(test_df)

print(f"\nTrain: {len(train_inputs)}")
print(f"Val: {len(val_inputs)}")
print(f"Test: {len(test_inputs)}")


Train: 554526
Val: 120934
Test: 118972


In [17]:
# сохранение предобработки
torch.save({
    'train_inputs': train_inputs,
    'train_targets': train_targets,
    'train_authors': train_authors,
    'train_categories': train_categories,
    'validate_inputs': val_inputs,
    'validate_targets': val_targets,
    'validate_authors': val_authors,
    'validate_categories': val_categories,
    'test_inputs': test_inputs,
    'test_targets': test_targets,
    'test_authors': test_authors,
    'test_categories': test_categories,
    'cnt_item': cnt_item,
    'cnt_author': cnt_author,
    'cnt_category': cnt_category,
    'item_id_to_idx': item_id_to_idx
}, 'preprocessed_data_exp.pt')